In [ ]:
import pandas as pd
from pathlib import Path

BASE        = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / "DECISIONS.md").exists())
ARQUIVO_IN  = BASE / "data/processed/pnadc_2023_anual.csv"
ARQUIVO_OUT = BASE / "data/processed/horas_por_setor_2023_corrigido.csv"

print("Imports OK")

In [ ]:
df = pd.read_csv(ARQUIVO_IN, dtype=str)
df["VD4031"] = pd.to_numeric(df["VD4031"], errors="coerce")

ocupados = df[
    (df["VD4002"].str.strip() == "1") &
    (df["VD4031"].notna()) &
    (df["VD4031"] > 0)
].copy()

setores = {
    "00": "Ativ. mal definidas",
    "10": "Agropecuária",
    "20": "Indústria geral",
    "30": "Construção",
    "40": "Comércio e rep.",
    "50": "Transp. e armaz.",
    "60": "Alojamento e alim.",
    "70": "Inf. e serv. prof.",
    "80": "Adm. públ. e educação",
    "81": "Saúde",
    "90": "Serv. domésticos",
}
ocupados["setor"] = ocupados["VD4010"].str.strip().map(setores)

print(f"Ocupados: {len(ocupados):,}")

In [ ]:
def calcular_metricas(grupo):
    h = grupo["VD4031"]
    n = len(h)
    return pd.Series({
        "n":                  n,
        "media":              round(h.mean(), 2),
        "mediana":            round(h.median(), 1),
        "p25":                round(h.quantile(0.25), 1),
        "p75":                round(h.quantile(0.75), 1),
        
        "pct_abaixo_40":      round((h < 40).mean() * 100, 2),
        "pct_40h_exatas":     round((h == 40).mean() * 100, 2),
        "pct_41_44h":         round(((h > 40) & (h <= 44)).mean() * 100, 2),  # CORRIGIDO
        "pct_acima_44h":      round((h > 44).mean() * 100, 2),
        
        "soma_faixas":        round(
            (h < 40).mean() * 100 +
            (h == 40).mean() * 100 +
            ((h > 40) & (h <= 44)).mean() * 100 +
            (h > 44).mean() * 100, 1
        ),
    })

df_metricas = (
    ocupados.groupby("setor")
    .apply(calcular_metricas)
    .sort_values("media", ascending=False)
)

print("\n── Métricas corrigidas por setor ────────────────────────")
print(df_metricas.to_string())

In [ ]:
print("\n── Verificação: soma das faixas (deve ser 100%) ─────────")
print(df_metricas["soma_faixas"].to_string())

In [ ]:
df_antigo = pd.read_csv(BASE / "data/processed/horas_por_setor_2023.csv", index_col=0)

print("\n── Comparação: pct_41_44h (antigo vs corrigido) ─────────")
print(f"{'Setor':<30} {'Antigo (incl. 40h)':>20} {'Corrigido (41-44h)':>20} {'Diferença':>12}")
print("-" * 85)

for setor in df_metricas.index:
    if setor in df_antigo.index:
        antigo   = df_antigo.loc[setor, "pct_40_44"]
        corrigido = df_metricas.loc[setor, "pct_41_44h"]
        diff     = corrigido - antigo
        print(f"{setor:<30} {antigo:>20.1f}% {corrigido:>20.1f}% {diff:>11.1f}%")

In [ ]:
df_metricas.to_csv(ARQUIVO_OUT)
print(f"\nSalvo: {ARQUIVO_OUT}")

print("\n── Resumo das faixas corrigidas (todos setores) ─────────")
print(df_metricas[["n", "media", "pct_abaixo_40", "pct_40h_exatas",
                    "pct_41_44h", "pct_acima_44h"]].to_string())